# 🚀 Code Review Chatbot — Üye 2: Model Eğitimi & RAG Pipeline
**YZM358 Doğal Dil İşleme Projesi**

Bu notebook adım adım çalıştırılmalıdır. Her hücreyi sırayla çalıştırın, çıktıyı kontrol edin.

In [ ]:
# ============================================================
# ADIM 1: Gerekli Kütüphaneleri Yükle
# ============================================================
print('Kütüphaneler yükleniyor...')
!pip install -q transformers==4.41.0 datasets faiss-cpu sacrebleu bert-score huggingface_hub sentencepiece
print('✅ Tüm kütüphaneler yüklendi!')
print('\n⚠️  ÖNEMLİ: Şimdi Runtime > Restart Session (Çalışma Zamanını Yeniden Başlat) yapın!')
print('Yeniden başlattıktan sonra bu hücreyi ATIP direkt 2. hücreye geçin.')

Kütüphaneler yükleniyor...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.6 MB/s eta 0:00:00
✅ Tüm kütüphaneler yüklendi!

⚠️  ÖNEMLİ: Şimdi Runtime > Restart Session (Çalışma Zamanını Yeniden Başlat) yapın!
Yeniden başlattıktan sonra bu hücreyi ATIP direkt 2. hücreye geçin.


In [ ]:
# ============================================================
# ADIM 2: Google Drive Bağla ve Veriyi Yükle
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
from datasets import load_dataset

# Drive'daki veri dosyasının yolunu buraya yaz
VERI_YOLU = "/content/drive/MyDrive/CodeReviewBot/data/codereviewer_clean/train_clean.jsonl"

if not os.path.exists(VERI_YOLU):
    raise FileNotFoundError(f'❌ Veri dosyası bulunamadı: {VERI_YOLU}\nLütfen yolu kontrol et!')

print(f'✅ Veri dosyası bulundu: {VERI_YOLU}')
print('Veri yükleniyor...')

raw_dataset = load_dataset('json', data_files={'train': VERI_YOLU}, split='train')

print(f'✅ Veri yüklendi! Toplam satır: {len(raw_dataset)}')
print(f'Sütunlar: {raw_dataset.column_names}')
print('\n📊 İlk 2 örnek:')
for i in range(2):
    print(f'--- Örnek {i+1} ---')
    print(f'  input: {str(raw_dataset[i]["input"])[:100]}...')
    print(f'  msg  : {str(raw_dataset[i]["msg"])[:100]}')

Mounted at /content/drive
✅ Veri dosyası bulundu: /content/drive/MyDrive/CodeReviewBot/data/codereviewer_clean/train_clean.jsonl
Veri yükleniyor...


Generating train split: 0 examples [00:00, ? examples/s]

✅ Veri yüklendi! Toplam satır: 48874
Sütunlar: ['input', 'oldf', 'msg', 'y']

📊 İlk 2 örnek:
--- Örnek 1 ---
  input: @@ -595,8 +595,10 @@ namespace Kratos
             array_1d<double, 3> b = ZeroVector(3);
          ...
  msg  : I assumed that for CrossProduct the values were inverted as well... Is that right?
--- Örnek 2 ---
  input: @@ -22,8 +22,13 @@
 For internal use only; no backwards-compatibility guarantees.
 """
 
+from __fut...
  msg  : I think we should we avoid `import six` for consistency with the approach followed elsewhere. What d


In [ ]:
# ============================================================
# ADIM 3: Veriyi Temizle (Bozuk/Boş Satırları Sil)
# ============================================================
print(f'Temizlik öncesi toplam satır: {len(raw_dataset)}')

def derin_temizlik(ornek):
    kod = str(ornek.get('input', '') or '')
    yorum = str(ornek.get('msg', '') or '')

    # Kural 1: Tamamen boş veya sadece boşluktan oluşanları at
    if not kod.strip() or not yorum.strip():
        return False

    # Kural 2: 'None', 'null', 'nan' değerlerini at
    if kod.lower() in ['none', 'null', 'nan'] or yorum.lower() in ['none', 'null', 'nan']:
        return False

    # Kural 3: Hafızayı patlatan devasa satırları at
    if len(kod) > 4000 or len(yorum) > 1000:
        return False

    # Kural 4: Çok kısa yorumları at (anlamsız 1-2 karakterlik yorumlar)
    if len(yorum.strip()) < 5:
        return False

    return True

temiz_dataset = raw_dataset.filter(derin_temizlik)

silinen = len(raw_dataset) - len(temiz_dataset)
print(f'🗑️  Silinen bozuk satır: {silinen}')
print(f'✅ Temiz kalan satır  : {len(temiz_dataset)}')

Temizlik öncesi toplam satır: 48874


Filter:   0%|          | 0/48874 [00:00<?, ? examples/s]

🗑️  Silinen bozuk satır: 19
✅ Temiz kalan satır  : 48855


In [ ]:
# ============================================================
# ADIM 4: Model ve Tokenizer'ı Yükle
# (Bozuk tokenizer.json dosyasını atlayarak)
# ============================================================
import torch
import os
from huggingface_hub import snapshot_download
from transformers import RobertaTokenizer, T5ForConditionalGeneration

MODEL_ADI = 'Salesforce/codet5p-220m'
TOKENIZER_KLASORU = '/content/temiz_tokenizer'

# Tokenizer dosyalarını indir ama bozuk olanları alma
print('Tokenizer dosyaları indiriliyor (bozuk JSON dosyaları hariç)...')
os.makedirs(TOKENIZER_KLASORU, exist_ok=True)
snapshot_download(
    repo_id=MODEL_ADI,
    local_dir=TOKENIZER_KLASORU,
    ignore_patterns=['*.bin', '*.safetensors', 'tokenizer.json', 'special_tokens_map.json']
)

# Varsa kalan bozuk dosyaları sil
for bad_file in ['tokenizer.json', 'special_tokens_map.json', 'added_tokens.json', 'tokenizer_config.json']:
    bad_path = os.path.join(TOKENIZER_KLASORU, bad_file)
    if os.path.exists(bad_path):
        os.remove(bad_path)
        print(f'  🗑️  Silindi: {bad_file}')

print('\nTokenizer yükleniyor...')
tokenizer = RobertaTokenizer.from_pretrained(TOKENIZER_KLASORU)
print(f'✅ Tokenizer yüklendi! Kelime haznesi: {len(tokenizer)} token')

print(f'\nModel yükleniyor: {MODEL_ADI}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = T5ForConditionalGeneration.from_pretrained(MODEL_ADI).to(device)
print(f'✅ Model yüklendi! Cihaz: {device.upper()}')

Tokenizer dosyaları indiriliyor (bozuk JSON dosyaları hariç)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'T5Tokenizer'. 
The class this function is called from is 'RobertaTokenizer'.


  🗑️  Silindi: added_tokens.json
  🗑️  Silindi: tokenizer_config.json

Tokenizer yükleniyor...
✅ Tokenizer yüklendi! Kelime haznesi: 32100 token

Model yükleniyor: Salesforce/codet5p-220m


config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/446M [00:00<?, ?B/s]

✅ Model yüklendi! Cihaz: CUDA


In [ ]:
# ============================================================
# ADIM 5: Veriyi Eğitim için Hazırla
# ============================================================
from datasets import DatasetDict

# Eğitim için veri boyutunu belirle
# Not: Tüm veriyi eğitmek 4+ saat sürer. Hızlı test için 10000 satır önerilir.
EGITIM_SATIRI = 10000  # GPU hakkı varsa artır
EVAL_SATIRI   = 1000

egitim_verisi = temiz_dataset.select(range(EGITIM_SATIRI))
eval_verisi   = temiz_dataset.select(range(EGITIM_SATIRI, EGITIM_SATIRI + EVAL_SATIRI))

print(f'Eğitim verisi : {len(egitim_verisi)} satır')
print(f'Eval verisi   : {len(eval_verisi)} satır')

# Tokenize fonksiyonu
def tokenize_fn(examples):
    # Giriş: "oldf" (eski kod) + "input" (yeni kod değişikliği)
    inputs = [
        f"Review this code change:\n[OLD]: {str(old)[:500]}\n[NEW]: {str(new)[:500]}"
        for old, new in zip(examples['oldf'], examples['input'])
    ]
    targets = [str(msg) for msg in examples['msg']]

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        padding='max_length',
        truncation=True
    )
    labels = tokenizer(
        text_target=targets,
        max_length=128,
        padding='max_length',
        truncation=True
    )

    # Padding token'larını -100 ile maskele (kritik!)
    labels['input_ids'] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels['input_ids']
    ]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print('Veri tokenize ediliyor...')
tokenized_train = egitim_verisi.map(tokenize_fn, batched=True, remove_columns=egitim_verisi.column_names)
tokenized_eval  = eval_verisi.map(tokenize_fn, batched=True, remove_columns=eval_verisi.column_names)
print(f'✅ Tokenize tamamlandı!')
print(f'  Eğitim: {len(tokenized_train)} örnek')
print(f'  Eval  : {len(tokenized_eval)} örnek')

Eğitim verisi : 10000 satır
Eval verisi   : 1000 satır
Veri tokenize ediliyor...


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

✅ Tokenize tamamlandı!
  Eğitim: 10000 örnek
  Eval  : 1000 örnek


In [ ]:
# ============================================================
# ADIM 6: Modeli Eğit (Fine-Tuning)
# ============================================================
from torch.optim import AdamW
from torch.utils.data import DataLoader
import torch

CIKTI_KLASORU = '/content/drive/MyDrive/CodeReviewBot/model_ciktisi'
os.makedirs(CIKTI_KLASORU, exist_ok=True)

tokenized_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_eval.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

train_loader = DataLoader(tokenized_train, batch_size=4, shuffle=True)
eval_loader  = DataLoader(tokenized_eval,  batch_size=4)

optimizer = AdamW(model.parameters(), lr=5e-5)

EPOCHS = 3
print(f'Eğitim başlıyor... ({len(train_loader)} adım x {EPOCHS} epoch)')

for epoch in range(EPOCHS):
    model.train()
    toplam_loss = 0
    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        toplam_loss += loss.item()

        if step % 200 == 0:
            print(f'  Epoch {epoch+1} | Adım {step}/{len(train_loader)} | Loss: {loss.item():.4f}')

    ort_loss = toplam_loss / len(train_loader)
    print(f'✅ Epoch {epoch+1} tamamlandı | Ortalama Loss: {ort_loss:.4f}')

model.save_pretrained(CIKTI_KLASORU + '/final_model')
tokenizer.save_pretrained(CIKTI_KLASORU + '/final_model')
print('✅ Model kaydedildi!')


Eğitim başlıyor... (2500 adım x 3 epoch)
  Epoch 1 | Adım 0/2500 | Loss: 5.8484
  Epoch 1 | Adım 200/2500 | Loss: 4.0340
  Epoch 1 | Adım 400/2500 | Loss: 3.3711
  Epoch 1 | Adım 600/2500 | Loss: 3.5309
  Epoch 1 | Adım 800/2500 | Loss: 3.2332
  Epoch 1 | Adım 1000/2500 | Loss: 3.6076
  Epoch 1 | Adım 1200/2500 | Loss: 3.9960
  Epoch 1 | Adım 1400/2500 | Loss: 2.6708
  Epoch 1 | Adım 1600/2500 | Loss: 2.4610
  Epoch 1 | Adım 1800/2500 | Loss: 3.5762
  Epoch 1 | Adım 2000/2500 | Loss: 3.3459
  Epoch 1 | Adım 2200/2500 | Loss: 3.6059
  Epoch 1 | Adım 2400/2500 | Loss: 3.8448
✅ Epoch 1 tamamlandı | Ortalama Loss: 3.5686
  Epoch 2 | Adım 0/2500 | Loss: 3.3952
  Epoch 2 | Adım 200/2500 | Loss: 2.8589
  Epoch 2 | Adım 400/2500 | Loss: 2.9816
  Epoch 2 | Adım 600/2500 | Loss: 3.7651
  Epoch 2 | Adım 800/2500 | Loss: 2.1024
  Epoch 2 | Adım 1000/2500 | Loss: 2.1263
  Epoch 2 | Adım 1200/2500 | Loss: 3.0394
  Epoch 2 | Adım 1400/2500 | Loss: 3.1220
  Epoch 2 | Adım 1600/2500 | Loss: 3.3540
  Ep

In [ ]:
# ============================================================
# ADIM 7: RAG Sistemi (FAISS) Kur
# ============================================================
import faiss
import numpy as np
import torch

print('FAISS (RAG) veritabanı oluşturuluyor...')
print('Bu işlem birkaç dakika sürebilir...')

RAG_BOYUTU = 2000  # Kaç örnek FAISS'e eklensin?
rag_verisi = temiz_dataset.select(range(RAG_BOYUTU))

# Kod örneklerini vektöre çevir
embeddings_list = []
corpus_data = []

model.eval()
for i, ornek in enumerate(rag_verisi):
    if i % 200 == 0:
        print(f'  İşlendi: {i}/{RAG_BOYUTU}')

    metin = str(ornek.get('input', ''))[:512]
    inputs = tokenizer(metin, return_tensors='pt', truncation=True, max_length=512).to(device)

    with torch.no_grad():
        outputs = model.encoder(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1).cpu().numpy()

    embeddings_list.append(embedding)
    corpus_data.append(ornek)

# FAISS indeksini oluştur
all_embeddings = np.vstack(embeddings_list).astype('float32')
boyut = all_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(boyut)
faiss_index.add(all_embeddings)

print(f'✅ FAISS indeksi hazır! {faiss_index.ntotal} vektör eklendi.')

ModuleNotFoundError: No module named 'faiss'

In [ ]:
# ============================================================
# ADIM 8: RAG + Model ile Kod İncele (TEST)
# ============================================================

def kod_incele(query_code, k=2, verbose=True):
    # 1. FAISS ile benzer örnekleri bul
    query_inp = tokenizer(query_code, return_tensors='pt', truncation=True, max_length=512).to(device)
    with torch.no_grad():
        query_emb = model.encoder(**query_inp).last_hidden_state.mean(dim=1).cpu().numpy().astype('float32')

    distances, indices = faiss_index.search(query_emb, k)
    benzer_ornekler = [corpus_data[idx] for idx in indices[0] if idx != -1]

    if verbose:
        print('🔍 RAG SİSTEMİ — Benzer Geçmiş İncelemeler:')
        print('='*60)
        for i, ornek in enumerate(benzer_ornekler):
            print(f'  [{i+1}] {str(ornek.get("msg", ""))[:120]}')
        print('='*60)

    # 2. Modele soruyu sor (eğitildiği formatla aynı!)
    prompt = f'Review this code change:\n[OLD]: \n[NEW]: {query_code[:500]}'
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=128,
            min_length=5,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    cevap = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return cevap


# ---- TEST KODLARI ----
test_kodlari = [
    ('Sıfıra bölme riski', 'def divide(a, b):\n    return a / b'),
    ('SQL Injection',       'def get_user(u):\n    query = "SELECT * FROM users WHERE name = " + u\n    return db.execute(query)'),
    ('Sonsuz recursive',   'def factorial(n):\n    return n * factorial(n-1)'),
    ('Temiz kod',          'def add(a, b):\n    return a + b'),
]

print('\n' + '='*60)
print('🧪 MODEL TEST SONUÇLARI')
print('='*60)

for baslik, kod in test_kodlari:
    print(f'\n[{baslik}]')
    print(f'Kod: {kod}')
    yorum = kod_incele(kod, verbose=True)
    print(f'Model: {yorum}')
    print('-'*60)


🧪 MODEL TEST SONUÇLARI

[Sıfıra bölme riski]
Kod: def divide(a, b):
    return a / b


NameError: name 'tokenizer' is not defined

In [ ]:
# ============================================================
# ADIM 9: BLEU Skoru ile Değerlendirme (Metrik)
# ============================================================
import sacrebleu

print('BLEU skoru hesaplanıyor...')

# 100 örnekle değerlendirme yap
test_veri = temiz_dataset.select(range(EGITIM_SATIRI + EVAL_SATIRI, EGITIM_SATIRI + EVAL_SATIRI + 100))

tahminler = []
referanslar = []

for i, ornek in enumerate(test_veri):
    if i % 20 == 0:
        print(f'  Test: {i}/100')

    tahmin = kod_incele(str(ornek['input']), verbose=False)
    tahminler.append(tahmin)
    referanslar.append(str(ornek['msg']))

# BLEU hesapla
bleu = sacrebleu.corpus_bleu(tahminler, [referanslar])
print(f'\n✅ BLEU Skoru: {bleu.score:.2f}')
print('\n📋 Örnek Tahmin vs Gerçek:')
for i in range(3):
    print(f'  Tahmin  : {tahminler[i]}')
    print(f'  Gerçek  : {referanslar[i]}')
    print()

BLEU skoru hesaplanıyor...
  Test: 0/100
  Test: 20/100
  Test: 40/100
  Test: 60/100
  Test: 80/100

✅ BLEU Skoru: 1.27

📋 Örnek Tahmin vs Gerçek:
  Tahmin  : I don't think we should use `from apache_beam.utils.counters` here.
  Gerçek  : Can we move this to the top? There don't seem to be any circular dependency issues.

  Tahmin  : I think this should be `None` instead of `None`.
  Gerçek  : rewording, it took me long time to understand it. (I'll propose something)

  Tahmin  : Can we rename this to `installInaboxCidService`?
  Gerçek  : Are these pure imports? I'm worried this will bring in a bunch of inabox files into the v0.js runtime.



In [ ]:
!pip install -q transformers bert-score datasets
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from transformers import RobertaTokenizer, T5ForConditionalGeneration
from datasets import load_dataset
import torch

MODEL_YOLU = '/content/drive/MyDrive/CodeReviewBot/model_ciktisi/final_model'
VERI_YOLU  = '/content/drive/MyDrive/CodeReviewBot/data/codereviewer_clean/train_clean.jsonl'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = RobertaTokenizer.from_pretrained(MODEL_YOLU)
model = T5ForConditionalGeneration.from_pretrained(MODEL_YOLU).to(device)
dataset = load_dataset('json', data_files=VERI_YOLU, split='train')
print(f"✅ Model yüklendi! Cihaz: {device.upper()}")


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

✅ Model yüklendi! Cihaz: CPU


In [ ]:
from bert_score import score as bert_score_fn

# 30 örnekle hızlı test (CPU'da da çalışır)
test_veri = dataset.select(range(48000, 48030))
tahminler, referanslar = [], []

for ornek in test_veri:
    prompt = f"Review this code change:\n[OLD]: {str(ornek['oldf'])[:300]}\n[NEW]: {str(ornek['input'])[:300]}"
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_length=128, num_beams=2)
    tahminler.append(tokenizer.decode(out[0], skip_special_tokens=True))
    referanslar.append(str(ornek['msg']))

P, R, F1 = bert_score_fn(tahminler, referanslar, lang="en", verbose=True)
print(f"\n✅ BERTScore Sonuçları:")
print(f"   Precision : {P.mean():.4f}")
print(f"   Recall    : {R.mean():.4f}")
print(f"   F1        : {F1.mean():.4f}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 89.88 seconds, 0.33 sentences/sec

✅ BERTScore Sonuçları:
   Precision : 0.8528
   Recall    : 0.8516
   F1        : 0.8514


In [ ]:
!pip install -q transformers faiss-cpu datasets

from google.colab import drive
drive.mount('/content/drive')

import torch, faiss, pickle, os
from transformers import RobertaTokenizer, T5ForConditionalGeneration
from datasets import load_dataset

MODEL_YOLU = '/content/drive/MyDrive/CodeReviewBot/model_ciktisi/final_model'
VERI_YOLU  = '/content/drive/MyDrive/CodeReviewBot/data/codereviewer_clean/train_clean.jsonl'
KAYIT_YOLU = '/content/drive/MyDrive/CodeReviewBot/model_ciktisi'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Cihaz: {device.upper()}')

tokenizer = RobertaTokenizer.from_pretrained(MODEL_YOLU)
model = T5ForConditionalGeneration.from_pretrained(MODEL_YOLU).to(device)
print('✅ Model yüklendi!')

dataset = load_dataset('json', data_files=VERI_YOLU, split='train')
rag_verisi = dataset.select(range(2000))
print('Veri yüklendi, FAISS oluşturuluyor...')

embeddings_list, corpus_data = [], []
model.eval()
for i, ornek in enumerate(rag_verisi):
    if i % 200 == 0: print(f'  {i}/2000')
    metin = str(ornek.get('input', ''))[:512]
    inputs = tokenizer(metin, return_tensors='pt', truncation=True, max_length=512).to(device)
    with torch.no_grad():
        emb = model.encoder(**inputs).last_hidden_state.mean(dim=1).cpu().numpy()
    embeddings_list.append(emb)
    corpus_data.append(ornek)

import numpy as np
all_emb = np.vstack(embeddings_list).astype('float32')
index = faiss.IndexFlatL2(all_emb.shape[1])
index.add(all_emb)

faiss.write_index(index, f'{KAYIT_YOLU}/faiss_index.bin')
with open(f'{KAYIT_YOLU}/corpus_data.pkl', 'wb') as f:
    pickle.dump(corpus_data, f)

print(f'\n✅ HER ŞEY KAYDEDILDI!')
print(f'  ├── final_model/     ✅')
print(f'  ├── faiss_index.bin  ✅')
print(f'  └── corpus_data.pkl  ✅')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 29.5 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cihaz: CPU


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

✅ Model yüklendi!
Veri yüklendi, FAISS oluşturuluyor...
  0/2000
  200/2000
  400/2000
  600/2000
  800/2000
  1000/2000
  1200/2000
  1400/2000
  1600/2000
  1800/2000

✅ HER ŞEY KAYDEDILDI!
  ├── final_model/     ✅
  ├── faiss_index.bin  ✅
  └── corpus_data.pkl  ✅


In [17]:
from google.colab import drive
drive.mount('/content/drive')

import json

metrikler = {
    "bleu":           1.27,
    "bertscore_p":    0.8528,
    "bertscore_r":    0.8516,
    "bertscore_f1":   0.8514,
    "egitim_loss": {
        "epoch1": 3.5686,
        "epoch2": 3.1015,
        "epoch3": 2.7506
    },
    "test_ornek_sayisi": 100
}

METRIK_YOLU = '/content/drive/MyDrive/CodeReviewBot/model_ciktisi/metrikler.json'
with open(METRIK_YOLU, 'w') as f:
    json.dump(metrikler, f, indent=2)

print("✅ Metrikler kaydedildi!")
print(json.dumps(metrikler, indent=2))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Metrikler kaydedildi!
{
  "bleu": 1.27,
  "bertscore_p": 0.8528,
  "bertscore_r": 0.8516,
  "bertscore_f1": 0.8514,
  "egitim_loss": {
    "epoch1": 3.5686,
    "epoch2": 3.1015,
    "epoch3": 2.7506
  },
  "test_ornek_sayisi": 100
}
